In [4]:
#ใช้ whisper ของ open ai รับ speech to text
!pip install openai-whisper pydub
!pip install SpeechRecognition PyAudio gTTS mutagen pandas
import whisper
from base64 import b64decode
import time
from pydub import AudioSegment

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 73.2 kB/s  0:07:35m0:00:0200:13
  error: subprocess-exited-with-error
  
  × Building wheel for PyAudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [27 lines of output]
      /private/var/folders/3j/dn3gq7jj1pg8vq9q_h_3kyfh0000gn/T/pip-build-env-8zh6hkng/overlay/lib/python3.12/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: MIT License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
             

In [ ]:
import time
import whisper
import io
from google.colab import output
from base64 import b64decode
from pydub import AudioSegment

# 1. Setup Model (โหลดทิ้งไว้ก่อนเริ่มจับเวลา)
print("กำลังโหลดโมเดล Whisper (base)...")
model = whisper.load_model("base")

def record_stt_benchmark():
    js = """
    async function recordAll() {
      // สร้าง UI
      const div = document.createElement('div');
      const button = document.createElement('button');
      const status = document.createElement('div');

      button.textContent = 'กดค้างเพื่อพูด (บันทึกเสียง & ทดสอบ)';
      button.style.padding = '15px';
      button.style.fontSize = '16px';
      button.style.cursor = 'pointer';
      button.style.borderRadius = '8px';

      status.style.marginTop = '10px';
      status.style.fontWeight = 'bold';

      div.appendChild(button);
      div.appendChild(status);
      document.body.appendChild(div);

      // เตรียมไมค์สำหรับ Whisper
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      // เตรียม Web Speech API
      const recognition = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
      recognition.lang = 'th-TH';
      recognition.interimResults = false;

      let webSpeechResult = "";
      recognition.onresult = (e) => { webSpeechResult = e.results[0][0].transcript; };

      return new Promise((resolve) => {
        recorder.ondataavailable = (e) => chunks.push(e.data);

        button.onmousedown = () => {
          chunks.length = 0;
          recorder.start();
          recognition.start();
          status.innerText = '🔴 กำลังฟังและบันทึกเสียง...';
          button.style.background = '#ffcccc';
        };

        button.onmouseup = () => {
          recorder.stop();
          recognition.stop();
          status.innerText = '⌛ ประมวลผลเสร็จแล้ว ส่งข้อมูลกลับไปที่ Python...';
          button.style.background = '#eeeeee';
        };

        recorder.onstop = async () => {
          const blob = new Blob(chunks);
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          reader.onloadend = () => {
            resolve({
              audioBase64: reader.result.split(',')[1],
              webSpeechTranscript: webSpeechResult
            });
          };
          stream.getTracks().forEach(track => track.stop());
        };
      });
    }
    """
    return output.eval_js(f'({js})()')

# --- เริ่มการทำงาน ---
print("\n[ระบบพร้อม] กรุณากดปุ่มเพื่อเริ่มทดสอบ...")
raw_data = record_stt_benchmark()

# 2. จัดการข้อมูลเสียง (Whisper Preparation)
audio_bytes = b64decode(raw_data['audioBase64'])
with open("audio.wav", "wb") as f:
    f.write(audio_bytes)

# 3. วัดเวลา Recording Time
audio_file = AudioSegment.from_file("audio.wav")
recording_time = len(audio_file) / 1000.0

# 4. วัดเวลา Whisper Inference
print("\n--- กำลังถอดความด้วย Whisper ---")
start_time = time.time()
whisper_result = model.transcribe("audio.wav", language="th")
inference_time = time.time() - start_time

# 5. คำนวณ RTF
rtf = inference_time / recording_time if recording_time > 0 else 0

# --- แสดงผลลัพธ์ใน Format เดียวกัน ---
print("\n" + "="*50)
print(f"{'สรุปผลการทดสอบ STT':^50}")
print("="*50)
print(f"เวลาที่ใช้พูด (Reference Time):   {recording_time:.2f} วินาที")
print(f"เวลาที่ Whisper ใช้ (Inference):  {inference_time:.2f} วินาที")
print(f"Real-time Factor (RTF):        {rtf:.2f}")
print("-" * 50)
print(f"ผลจาก OpenAI Whisper:  '{whisper_result['text'].strip()}'")
print("="*50)

if rtf < 1:
    print("Whisper ทำงานเร็วกว่า Real-time")
else:
    print("Whisper ทำงานช้ากว่า Real-time (อาจเกิดความล่าช้าใน Virtual AI)")

กำลังโหลดโมเดล Whisper (base)...


100%|███████████████████████████████████████| 139M/139M [00:11<00:00, 13.1MiB/s]



[ระบบพร้อม] กรุณากดปุ่มเพื่อเริ่มทดสอบ...


NameError: name 'output' is not defined

### ใช้ Web speech API ในการทำ speech to text
ระบบจำเสียงที่ ติดมากับ Browser (Chrome/Edge) ของ Google

In [ ]:

from google.colab import output

def record_web_speech_test():
    js = """
    async function recordWebSpeech() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      const status = document.createElement('div');

      button.textContent = 'กดค้างเพื่อพูด (วัดผล Web Speech API)';
      button.style.padding = '15px';
      button.style.fontSize = '16px';
      button.style.cursor = 'pointer';

      status.style.marginTop = '10px';
      status.style.color = '#555';

      div.appendChild(button);
      div.appendChild(status);
      document.body.appendChild(div);

      let startTime, stopTime, finalTime;
      let transcript = "";

      const recognition = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
      recognition.lang = 'th-TH';
      recognition.interimResults = false; // เอาแค่ผลลัพธ์สุดท้ายเพื่อความแม่นยำ
      recognition.continuous = false;

      return new Promise((resolve) => {
        // เมื่อได้ผลลัพธ์สุดท้ายจาก Google
        recognition.onresult = (event) => {
          finalTime = performance.now(); // เวลาที่ได้ Text กลับมาจริง
          transcript = event.results[0][0].transcript;
        };

        button.onmousedown = () => {
          transcript = "";
          status.innerText = '🔴 กำลังฟัง... (พูดได้เลย)';
          button.style.background = '#ffcccc';
          startTime = performance.now(); // เริ่มนับเวลาที่คนพูด
          recognition.start();
        };

        button.onmouseup = () => {
          stopTime = performance.now(); // จบนับเวลาที่คนพูด
          status.innerText = '⌛ กำลังรอผลลัพธ์จาก Google...';
          button.style.background = '#eeeeee';
          recognition.stop();
        };

        // ส่งข้อมูลกลับไปยัง Python เมื่อประมวลผลเสร็จ
        recognition.onend = () => {
          resolve({
            transcript: transcript,
            referenceTime: (stopTime - startTime) / 1000, // เวลาที่คนใช้พูด
            totalLatency: (finalTime - stopTime) / 1000   // เวลาที่ Google ใช้คิดหลังพูดจบ
          });
        };
      });
    }
    """
    return output.eval_js(f'({js})()')

# --- เริ่มการทดสอบ ---
print("ระบบพร้อมทดสอบ Web Speech API (เฉพาะ Chrome/Edge เท่านั้น)")
data = record_web_speech_test()

# ดึงค่าที่วัดได้
ref_time = data['referenceTime']
inf_time = data['totalLatency']
rtf = inf_time / ref_time if ref_time > 0 else 0

print("\n" + "="*40)
print(f"สิ่งที่userพูด: '{data['transcript']}'")
print("-" * 40)
print(f"Reference Time(เวลาที่พูดจริง): {ref_time:.2f} วินาที")
print(f"Inference Time(Google ใช้คิด): {inf_time:.2f} วินาที")
print(f"Real-time Factor: {rtf:.2f}")
print("="*40)

if inf_time < 0.5:
    print("ผลสรุป: ตอบสนองไวมาก (Low Latency) เหมาะสำหรับ Virtual AI")
else:
    print("ผลสรุป: มีความล่าช้าเล็กน้อยจากอินเทอร์เน็ต")

ระบบพร้อมทดสอบ Web Speech API (เฉพาะ Chrome/Edge เท่านั้น)

สิ่งที่userพูด: 'ขอข้อมูลค่าเทอมหน่อย'
----------------------------------------
Reference Time(เวลาที่พูดจริง): 3.51 วินาที
Inference Time(Google ใช้คิด): 0.12 วินาที
Real-time Factor: 0.03
ผลสรุป: ตอบสนองไวมาก (Low Latency) เหมาะสำหรับ Virtual AI
